# 토픽 간 유사도 분석

두 가지 방법으로 토픽 간 유사도를 측정합니다.

| 방법 | 사용 데이터 | 의미 |
|------|------------|------|
| **c-TF-IDF 유사도** | `step6_8_ctfidf.pkl` | 토픽을 대표하는 키워드가 얼마나 겹치는지 |
| **Embedding 유사도** | `step3_1_embeddings.npy` + `step5_1_cluster_labels.npy` | 실제 리뷰 문장의 의미가 얼마나 가까운지 |

두 방법 모두 유사도가 높은 토픽 쌍 → 병합 또는 상위 라벨 묶음 후보

In [1]:
from google.colab import drive

import os
import time
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/sparta/tp4/review_analysis/data/'

print(f'OUTPUT_DIR: {OUTPUT_DIR}')

Mounted at /content/drive
OUTPUT_DIR: /content/drive/MyDrive/sparta/tp4/review_analysis/data/


## 방법 1. c-TF-IDF 유사도

토픽별 c-TF-IDF 벡터 간 cosine similarity.
키워드가 비슷하게 분포된 토픽끼리 높게 나옵니다.

In [3]:
# 저장된 c-TF-IDF 행렬 로드
with open(OUTPUT_DIR + 'step6_8_ctfidf.pkl', 'rb') as f:
    ctfidf_matrix, words, topic_id_order = pickle.load(f)

ctfidf_array = ctfidf_matrix.toarray()
print(f'c-TF-IDF 행렬: {ctfidf_array.shape}  (토픽 수 x 단어 수)')
print(f'토픽 수: {len(topic_id_order)}개')

c-TF-IDF 행렬: (68, 14651)  (토픽 수 x 단어 수)
토픽 수: 68개


In [4]:
# 토픽 간 cosine similarity 계산
sim_matrix = cosine_similarity(ctfidf_array)  # (n_topics, n_topics)

# 자기 자신(대각선) 제거 → 상삼각 행렬만
n = len(topic_id_order)
pairs = []
for i in range(n):
    for j in range(i + 1, n):
        pairs.append({
            'topic_a': topic_id_order[i],
            'topic_b': topic_id_order[j],
            'ctfidf_sim': round(float(sim_matrix[i, j]), 4),
        })

df_ctfidf_sim = pd.DataFrame(pairs).sort_values('ctfidf_sim', ascending=False)

print('[c-TF-IDF 유사도 상위 30쌍]')
print(df_ctfidf_sim.head(30).to_string(index=False))

[c-TF-IDF 유사도 상위 30쌍]
 topic_a  topic_b  ctfidf_sim
      27       32      0.6794
      51       53      0.6713
      30       31      0.6648
      57       64      0.6645
      13       15      0.6416
       6       39      0.6045
      43       67      0.5991
      43       64      0.5988
      47       49      0.5964
      12       13      0.5886
      44       46      0.5825
      29       30      0.5818
      64       66      0.5755
      29       31      0.5731
      57       66      0.5724
      46       61      0.5719
      44       61      0.5660
      43       57      0.5627
      18       19      0.5566
       2       63      0.5428
      15       46      0.5401
      13       14      0.5370
      10       11      0.5314
       2        6      0.5302
      51       64      0.5297
      12       33      0.5284
      14       15      0.5279
      55       64      0.5264
      47       50      0.5236
      13       46      0.5215


In [5]:
# 임계값 이상인 쌍만 추출 (기본 0.5 — 조정 가능)
THRESHOLD = 0.5

high_sim = df_ctfidf_sim[df_ctfidf_sim['ctfidf_sim'] >= THRESHOLD].copy()
print(f'c-TF-IDF 유사도 >= {THRESHOLD} 인 토픽 쌍: {len(high_sim)}개')
print(high_sim.to_string(index=False))

c-TF-IDF 유사도 >= 0.5 인 토픽 쌍: 41개
 topic_a  topic_b  ctfidf_sim
      27       32      0.6794
      51       53      0.6713
      30       31      0.6648
      57       64      0.6645
      13       15      0.6416
       6       39      0.6045
      43       67      0.5991
      43       64      0.5988
      47       49      0.5964
      12       13      0.5886
      44       46      0.5825
      29       30      0.5818
      64       66      0.5755
      29       31      0.5731
      57       66      0.5724
      46       61      0.5719
      44       61      0.5660
      43       57      0.5627
      18       19      0.5566
       2       63      0.5428
      15       46      0.5401
      13       14      0.5370
      10       11      0.5314
       2        6      0.5302
      51       64      0.5297
      12       33      0.5284
      14       15      0.5279
      55       64      0.5264
      47       50      0.5236
      13       46      0.5215
      38       63      0.5200
       2

## 방법 2. Embedding 유사도

토픽별 리뷰 임베딩의 평균 벡터(centroid) 간 cosine similarity.
실제 리뷰 문장의 의미 공간에서 얼마나 가까운지를 측정합니다.

> 주의: `01_embeddings.npy`가 ~1.78GB로 크기 때문에 로드에 시간이 걸립니다.

In [6]:
print('embeddings 로드 중...')
t0 = time.time()
embeddings = np.load(OUTPUT_DIR + 'step3_1_embeddings.npy')
cluster_labels = np.load(OUTPUT_DIR + 'step5_1_cluster_labels.npy')
print(f'완료: {time.time()-t0:.1f}초')
print(f'embeddings: {embeddings.shape}')
print(f'cluster_labels: {cluster_labels.shape}')

embeddings 로드 중...
완료: 26.9초
embeddings: (598301, 768)
cluster_labels: (598301,)


In [7]:
# 토픽별 centroid 계산 (outlier -1 제외)
print('토픽별 centroid 계산 중...')
centroids = []
for tid in topic_id_order:
    mask = (cluster_labels == tid)
    centroid = embeddings[mask].mean(axis=0)
    centroids.append(centroid)

centroids = np.vstack(centroids)  # (n_topics, 768)
print(f'centroid 행렬: {centroids.shape}')

토픽별 centroid 계산 중...
centroid 행렬: (68, 768)


In [8]:
# centroid 간 cosine similarity 계산
emb_sim_matrix = cosine_similarity(centroids)

emb_pairs = []
for i in range(n):
    for j in range(i + 1, n):
        emb_pairs.append({
            'topic_a': topic_id_order[i],
            'topic_b': topic_id_order[j],
            'emb_sim': round(float(emb_sim_matrix[i, j]), 4),
        })

df_emb_sim = pd.DataFrame(emb_pairs).sort_values('emb_sim', ascending=False)

print('[Embedding 유사도 상위 30쌍]')
print(df_emb_sim.head(30).to_string(index=False))

[Embedding 유사도 상위 30쌍]
 topic_a  topic_b  emb_sim
      57       66   0.9439
      27       28   0.9400
      61       62   0.9300
      41       67   0.9296
      27       32   0.9283
      46       61   0.9269
      26       27   0.9267
      44       61   0.9267
      54       67   0.9225
      18       19   0.9208
      64       66   0.9181
      47       50   0.9178
      53       67   0.9171
      23       24   0.9170
      51       53   0.9156
      43       67   0.9131
      54       65   0.9113
      59       62   0.9104
      44       46   0.9097
      39       40   0.9092
      52       53   0.9075
      65       67   0.9072
      51       64   0.9066
      51       52   0.9064
      47       49   0.9053
      59       61   0.9042
      30       31   0.9039
      12       33   0.9038
      40       45   0.9033
       9       10   0.9022


## 방법 1 + 2 통합: 두 유사도 모두 높은 쌍

- c-TF-IDF 유사도가 높다 → 키워드가 비슷
- Embedding 유사도가 높다 → 리뷰 문장의 의미가 비슷

**둘 다 높은 쌍**이 병합 또는 상위 라벨 묶음의 가장 강력한 후보입니다.

In [9]:
# 두 유사도 병합
df_combined = pd.merge(df_ctfidf_sim, df_emb_sim, on=['topic_a', 'topic_b'])

# 둘 다 임계값 이상인 쌍
CTFIDF_THRESHOLD = 0.4
EMB_THRESHOLD    = 0.8

strong_candidates = df_combined[
    (df_combined['ctfidf_sim'] >= CTFIDF_THRESHOLD) &
    (df_combined['emb_sim']   >= EMB_THRESHOLD)
].sort_values('ctfidf_sim', ascending=False)

print(f'병합/묶음 강력 후보 (c-TF-IDF >= {CTFIDF_THRESHOLD}, Embedding >= {EMB_THRESHOLD}): {len(strong_candidates)}쌍')
print(strong_candidates.to_string(index=False))

병합/묶음 강력 후보 (c-TF-IDF >= 0.4, Embedding >= 0.8): 116쌍
 topic_a  topic_b  ctfidf_sim  emb_sim
      27       32      0.6794   0.9283
      51       53      0.6713   0.9156
      30       31      0.6648   0.9039
      57       64      0.6645   0.9020
      13       15      0.6416   0.8495
       6       39      0.6045   0.8553
      43       67      0.5991   0.9131
      43       64      0.5988   0.8268
      47       49      0.5964   0.9053
      12       13      0.5886   0.8860
      44       46      0.5825   0.9097
      29       30      0.5818   0.8657
      64       66      0.5755   0.9181
      29       31      0.5731   0.8724
      57       66      0.5724   0.9439
      46       61      0.5719   0.9269
      44       61      0.5660   0.9267
      43       57      0.5627   0.8949
      18       19      0.5566   0.9208
      13       14      0.5370   0.8421
      10       11      0.5314   0.8974
      51       64      0.5297   0.9066
      12       33      0.5284   0.9038
      14  

In [10]:
# 키워드와 함께 출력 (검토용)
kw_df = pd.read_csv(OUTPUT_DIR + 'step6_1_topic_keywords.csv')

def get_keywords(topic_id, top_n=8):
    kw = kw_df[kw_df['topic_id'] == topic_id].sort_values('rank')
    return ', '.join(kw['word'].tolist()[:top_n])

print('=' * 80)
print('[강력 후보 키워드 비교]')
print('=' * 80)
for _, row in strong_candidates.iterrows():
    ta, tb = int(row['topic_a']), int(row['topic_b'])
    print(f'\nT{ta:>3} vs T{tb:>3}  (c-TF-IDF: {row["ctfidf_sim"]:.3f}, Emb: {row["emb_sim"]:.3f})')
    print(f'  T{ta:>3}: {get_keywords(ta)}')
    print(f'  T{tb:>3}: {get_keywords(tb)}')

[강력 후보 키워드 비교]

T 27 vs T 32  (c-TF-IDF: 0.679, Emb: 0.928)
  T 27: 가을, 핏 가을, 적당 가을, 두께 가을, 가을 겨울, 봄가을, 가을 두께감, 오버핏 가을
  T 32: 가을, 여름 가을, 가을 겨울, 핏 가을, 두께감 가을, 적당 가을, 재질 가을, 두께 가을

T 51 vs T 53  (c-TF-IDF: 0.671, Emb: 0.916)
  T 51: 가격 퀄리티, 핏 가격, 퀄리티, 가격 대비, 대비, 가성비 최고, 핏 가성비, 가격 비하
  T 53: 가성비 편하, 저렴하, 편하 가격, 가격 저렴하, 저렴, 가격 대비, 무난 가격, 가성비 무난

T 30 vs T 31  (c-TF-IDF: 0.665, Emb: 0.904)
  T 30: 여름 시원, 여름, 재질 여름, 원단 여름, 여름 편하, 소재 여름, 두께 여름, 여름 색감
  T 31: 여름 시원, 여름 편하, 여름, 핏 여름, 여름 동안, 재질 여름, 사이즈 여름, 가성비 여름

T 57 vs T 64  (c-TF-IDF: 0.664, Emb: 0.902)
  T 57: 사이즈 적당, 사진 똑같, 번창, 색감 사이즈, 핏 색감, 똑같, 사이즈 색상, 사이즈 디자인
  T 64: 워싱, 화면, 똑같, 워싱 들어가, 색감 화면, 화면 똑같, 사진 똑같, 프린팅

T 13 vs T 15  (c-TF-IDF: 0.642, Emb: 0.850)
  T 13: 기모 따뜻, 기모, 기모 겨울, 핏 기모, 기모 들어가, 오버핏 기모, 사이즈 기모, 편하 기모
  T 15: 후드티, 후드, 후드집업, 기본 후드티, 후드티 핏, 오버핏 후드티, 후드 핏, 후드티 필요

T  6 vs T 39  (c-TF-IDF: 0.605, Emb: 0.855)
  T  6: 바지, 팬츠, 바지 핏, 청바지, 바지 편하, 카고팬츠, 벌룬팬츠, 와이드
  T 39: 허리, 스트링, 허벅지, 벨트, 밴딩, 수선, 다리, 허리 벨트

T 43 vs T 67  (c-TF-IDF:

In [11]:
# 결과 저장
df_combined.to_csv(OUTPUT_DIR + 'step6_14_topic_similarity.csv', index=False, encoding='utf-8-sig')
strong_candidates.to_csv(OUTPUT_DIR + 'step6_15_merge_candidates.csv', index=False, encoding='utf-8-sig')

print('저장 완료:')
print('  step6_14_topic_similarity.csv   (전체 토픽 쌍 유사도)')
print('  step6_15_merge_candidates.csv   (병합/묶음 강력 후보)')

저장 완료:
  step6_14_topic_similarity.csv   (전체 토픽 쌍 유사도)
  step6_15_merge_candidates.csv   (병합/묶음 강력 후보)
